# Off-Design Performance
### Roadrunner propulsion reverse-engineering study · notebook 8 of 13

**Purpose.** Run the required engine of notebook 7 away from its design point in pyCycle: over
altitude, Mach number and day at maximum power, and over throttle at the mission's own
conditions. It answers:

> *How does thrust fall with altitude and speed, how much worse does fuel consumption get at
> part throttle, and can the engine throttle down far enough to land?*

The engine is the notebook-9 design point (hot-day sea-level static, 542 N, anchored $T_{t4}$);
the design fixes the map scalars and the nozzle throat area (scaled generic maps,
ADR-0015). Off design, shaft speed balances shaft power and air flow balances the throat area.
The throttle is the turbine inlet temperature (maximum power: its design value) or the net
thrust. Thrust is **uninstalled** (ADR-0011). Fuel flow is the ideal value divided by the median
lumped combustion efficiency of the anchored technology level, held constant with throttle.
Points below the shaft-speed validity limit of `config/off_design.yaml` are flagged.

This notebook replaces the algebraic lapse and constant TSFC of the preliminary pair
(notebooks 5, 6) with the engine's own; notebooks 9 and 10 repeat those analyses on its results.

---
## 0 · Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # bare-checkout fallback

import yaml

from roadrunner_re import off_design as od
from roadrunner_re.analysis_inputs import load_analysis_inputs
from roadrunner_re.cycle.design import DesignInputs, size_design_point
from roadrunner_re.cycle.turbojet import FlightCondition, load_cycle_settings
from roadrunner_re.design_point import solve_design_point
from roadrunner_re.notebook import nb_setup
from roadrunner_re.plots.off_design import lapse_figure, part_throttle_figure

paths = nb_setup()
analysis = load_analysis_inputs()
design_point = solve_design_point(analysis)
settings = load_cycle_settings()
config = od.load_off_design_config()
anchored = yaml.safe_load((paths.out / "validation.yaml").read_text(encoding="utf-8"))["anchored_technology"]
hot = analysis.mission.dT_isa_K.value
design_inputs = DesignInputs(design_point.launch_thrust_per_engine_N, settings.pressure_ratio_when_unpublished.value,
                             anchored["Tt4_K"]["median"], FlightCondition(0.0, 0.0, hot), settings.nominal_losses())
design = size_design_point(settings, design_inputs)       # the notebook-9 engine
eta = anchored["combustion_efficiency"]["median"]
print(f"design: {design.thrust_N:.1f} N at hot-day sea-level static, {design.air_mass_flow_kg_s:.3f} kg/s, Tt4 {design_inputs.Tt4_K:.0f} K")
print(f"combustion efficiency (median, n={anchored['combustion_efficiency']['n']}): {eta:.3f}")

---
## 1 · Run the study

About a hundred pyCycle off-design points; this takes a couple of minutes.

In [ ]:
study = od.run_study(analysis, settings, config, design_inputs, design, design_point, eta)
ref = study.reference
print(f"reference (standard day, sea-level static, maximum power): {ref.thrust_N:.1f} N, shaft speed {ref.speed_fraction:.3f}")
print(f"unconverged or below the validity limit: {sum(not r['valid'] for r in study.lapse_rows + study.part_rows)} of {len(study.lapse_rows) + len(study.part_rows)}")

---
## 2 · Thrust lapse at maximum power

Thrust at the design turbine inlet temperature, divided by the standard-day sea-level static
thrust (the algebraic lapse of ADR-0029 is defined the same way). The dashed lines are that
algebraic lapse.

In [ ]:
fig = lapse_figure(study, paths.figures / "off_design_lapse.png")
cmp = od.off_design_handoff(study, analysis, design_point)["comparison_with_algebraic_lapse"]
print(f"cycle against algebraic lapse over {cmp['points']} points: mean |difference| {cmp['mean_abs_relative_difference']:.1%}, max {cmp['max_abs_relative_difference']:.1%}")
print()
print(f"{'alt [m]':>8s}{'Mach':>6s}{'cycle':>8s}{'algebraic':>11s}{'speed':>8s}   (ISA {hot:+.0f} K)")
for r in study.lapse_rows:
    if r["dT_isa_K"] == hot:
        print(f"{r['altitude_m']:8.0f}{r['mach']:6.1f}{r['lapse']:8.3f}{r['algebraic_lapse']:11.3f}{r['speed_fraction']:8.3f}")

---
## 3 · Part-throttle fuel consumption

At the mission's dash, loiter and return conditions and at sea-level static, thrust is stepped
down from its maximum at that condition. The loiter is the case the preliminary mission analysis
was expected to understate (OP4).

In [ ]:
fig = part_throttle_figure(study, paths.figures / "off_design_part_throttle.png")
print(f"{'condition':18s}{'thrust frac':>12s}{'SFC [kg/(N h)]':>16s}{'x full power':>14s}{'speed':>8s}{'Tt4 [K]':>9s}")
for label in dict.fromkeys(r["condition"] for r in study.part_rows):
    rows = sorted((r for r in study.part_rows if r["condition"] == label), key=lambda r: -r["thrust_fraction"])
    full = rows[0]["sfc_kg_N_s"]
    for r in rows:
        if r["thrust_fraction"] in (1.0, 0.6, 0.3):
            print(f"{label:18s}{r['thrust_fraction']:12.1f}{r['sfc_kg_N_s'] * 3600:16.3f}{r['sfc_kg_N_s'] / full:14.2f}{r['speed_fraction']:8.3f}{r['Tt4_K']:9.0f}")

In [ ]:
cmp = od.loiter_comparison(analysis, study, design_point)
print(f"loiter needs {cmp['loiter_thrust_fraction']:.0%} of the maximum thrust at its condition (mean mass, drag of the hand-estimate polar)")
print(f"SFC at that throttle  {cmp['sfc_cycle_kg_N_s'] * 3600:.3f} kg/(N h)  (cycle, real)")
print(f"preliminary constant  {cmp['sfc_preliminary_kg_N_s'] * 3600:.3f} kg/(N h)  (catalogue median)")
print(f"ratio {cmp['ratio_cycle_to_preliminary']:.2f}: the preliminary loiter fuel is scaled by about this factor in notebook 10")

---
## 4 · Landing thrust floor (OP2)

Landing needs thrust equal to the weight at the start of the landing segment, shared between the
engines, at sea level on the mission's hot day. Can the engine throttle that far down inside the
valid speed range?

In [ ]:
L = study.landing
print(f"required thrust per engine {L['required_thrust_per_engine_N']:.0f} N = {L['fraction_of_max_thrust']:.0%} of the maximum available there")
print(f"shaft speed {L['speed_fraction']:.3f} of design (limit {config.min_speed_fraction}), Tt4 {L['Tt4_K']:.0f} K, valid: {L['valid']}")
for label in dict.fromkeys(r["condition"] for r in study.part_rows):
    floor = min(r["thrust_fraction"] for r in study.part_rows if r["condition"] == label and r["valid"])
    print(f"lowest valid throttle at {label:18s}: {floor:.2f} of the maximum thrust")

---
## 5 · Handoff

In [ ]:
handoff = od.write_off_design_handoff(od.off_design_handoff(study, analysis, design_point), paths.out / "off_design.yaml")
print(f"wrote {handoff.relative_to(paths.repo)}")

---
## Findings

- **The algebraic lapse of the preliminary analysis is close but optimistic in flight.** Over the
  30-point grid the cycle's thrust differs from it by 6 % on average and 10.5 % at most. Static
  thrust agrees (0.885 against 0.882 at sea level on the hot day); at Mach 0.3 to 0.6 the cycle
  gives 6 to 9 % less, because ram drag grows faster than the algebraic form allows.
- **Part-throttle fuel consumption is flat to about 60 % thrust and then climbs**: 2 to 5 % worse
  at 60 % thrust, 28 to 36 % worse at 30 % (the OP1 against OP4 tension of ADR-0009, visible
  in the model's own numbers).
- **The loiter sits near the bad end of that curve.** It needs about 28 % of the maximum thrust at
  its condition, where the real SFC is 0.224 kg/(N h) against the 0.130 of the preliminary
  analysis, a factor of 1.73. About 1.29 of it is the flight condition at full power and the
  rest is part load. The preliminary loiter fuel was therefore understated, as expected; notebook
  12 quantifies it.
- **Landing is comfortable**: hovering at landing weight needs 66 % of the maximum thrust on the hot
  day, with shaft speed 0.93 of design. The model's throttle floor at sea-level static is about
  30 % of maximum thrust (at 25 % the speed falls below the validity limit, at 20 % it does not
  converge), so the thrust floor of OP2 is met with margin, but the engine's real minimum stable
  thrust is a datasheet question, not this model's.
- **Standing findings:** generic axial maps stand in for a centrifugal compressor; combustion
  efficiency is constant with throttle, so part-throttle fuel flow is, if anything, understated;
  thrust is uninstalled; the landing check is steady only, with no throttle response.